## 🎯 场景输入：

假设你是工厂调度员：

订单：  
- 可乐 10000瓶（3天内）  
- 橙汁 8000瓶（2天内）  
- 牛奶 5000瓶（1天内）  
  
机器：  
- 生产线A（饮料）  
- 生产线B（乳制品）  
  
约束：  
- 牛奶→果汁必须清洗（2小时）  
- 同类产品连续生产更高效

## STtP1:数据建模

order：
bot:

⚙️ Step 2：APS引擎（算法层）
核心逻辑：

目标：
- 最小延误
- 最小切换成本
- 最大产能利用率

现实 APS核心能力：

有限产能排产
多约束优化
瓶颈优先

🤖 Step 3：AI Agent 接管“决策层”（重点）

现在进入真正牛逼的部分👇

🧠 Agent 1：需求理解（Planner Agent）

输入：

老板说：优先利润高 + 别浪费

输出：

{
  "priority": ["profit", "due_date"],
  "penalty_late": 0.8,
  "penalty_changeover": 0.6
}

👉 把“人话” → “优化参数”

🤖 Agent 2：排产决策（Scheduler Agent）

调用：

schedule = APS_engine.solve(constraints)

🧠 Agent 3：解释（Explain Agent）

输出：

建议：
1. 先生产牛奶（最紧急）
2. 再生产橙汁（避免重复清洗）
3. 最后可乐（延误风险最低）

👉 传统APS做不到解释
👉 Agent可以

🔥 Agent 4：异常处理（关键）

现实一定会发生：

机器坏了
原料延迟

👉 Agent做：

机器A故障 → 自动重新排产

📌 研究里已经验证：

多Agent可以在供应链中快速重新调度

[ Chat UI ]
     ↓
[ AI Agent Layer ]
     ↓
[ APS Solver (OR-Tools) ]
     ↓
[ PostgreSQL / ERP / MES ]

In [ ]:
from pydantic import BaseModel
from pydantic_ai import Agent
class Order(BaseModel):
    product: str # 产品名称
    quantity: int # 数量
    due_date: int # 截止日期
    type: str  # beverage / dairy

class Machine(BaseModel):
    id: str # 机器ID
    supported_type: str  # 支持的产品类型
    capacity_per_hour: int  # 每小时产量

class Constraint(BaseModel):
    changeover_time: dict(str,int)  # ("milk","juice") -> 2h

PlannerAgent = Agent(
    'openrouter:xiaomi/mimo-v2-flash',
    tools=[Order, Machine, Constraint],
    verbose=True,
    max_tokens=1000,
    temperature=0.5,
)




In [23]:
from pydantic_ai.models.openrouter import OpenRouterModelSettings
from dataclasses import replace
from pydantic_ai.models.openrouter import OpenRouterProvider
base_profile = OpenRouterProvider.model_profile("xiaomi/mimo-v2-flash")
profile = replace(
    base_profile,
    supports_json_schema_output=True,
    supports_json_object_output=True,
    default_structured_output_mode="native",
)
settings = OpenRouterModelSettings(
    temperature=0.0,
    max_tokens=10000,
    top_p=1.0, #
    frequency_penalty=0.0, #（频率惩罚）
    presence_penalty=0.0, #（存在惩罚）
    openrouter_provider={"require_parameters": True},
)


In [ ]:
from __future__ import annotations
import logfire
logfire.configure()
logfire.instrument_pydantic_ai()

from enum import Enum
from typing import Literal
import json
from pydantic import BaseModel, Field
from pydantic import field_validator
from pydantic import model_validator
from pydantic_ai import Agent
from pydantic_ai.mcp import MCPServerStreamableHTTP

class OrderPriority(str, Enum): # 订单优先级
    deadline = "deadline" # 截止日期
    profit = "profit" # 利润
    inventory = "inventory" # 库存
    setup = "setup" # 换线
    throughput = "throughput" # 吞吐量

class ConstraintType(str, Enum):
    machine_capacity = "machine_capacity" # 机器容量
    changeover = "changeover" # 换线
    due_date = "due_date" # 截止日期
    inventory_limit = "inventory_limit" # 库存限制
    machine_compatibility = "machine_compatibility" # 机器兼容性
    cleaning_required = "cleaning_required" # 清洗要求
    batch_size = "batch_size" # 批量大小
    frozen_sequence = "frozen_sequence" # 冻结序列

class ObjectiveWeights(BaseModel): # 目标权重 0-1 之间
    minimize_lateness: float = Field(default=0.0, ge=0.0, le=1.0) # 最小化延迟
    maximize_profit: float = Field(default=0.0, ge=0.0, le=1.0) # 最大化利润
    minimize_inventory: float = Field(default=0.0, ge=0.0, le=1.0) # 最小化库存
    minimize_changeover: float = Field(default=0.0, ge=0.0, le=1.0) # 最小化换线
    maximize_throughput: float = Field(default=0.0, ge=0.0, le=1.0) # 最大化吞吐量
    @model_validator(mode="after")
    def validate_weights(self) -> "ObjectiveWeights":
        total = (
            self.minimize_lateness
            + self.maximize_profit
            + self.minimize_inventory
            + self.minimize_changeover
            + self.maximize_throughput
        )
        if total <= 0:
            raise ValueError("At least one objective weight must be greater than 0.")
        if total > 1.001:
            raise ValueError("Objective weights must sum to <= 1.0.")
        return self

class Constraint(BaseModel):
    type: ConstraintType # 约束类型
    description: str = Field(min_length=3) # 描述
    params: dict = Field(default_factory=dict) # 参数

class SolverConfig(BaseModel):
    horizon_hours: int = Field(default=72, ge=1, le=24 * 90) # 时间范围
    time_limit_seconds: int = Field(default=10, ge=1, le=300) # 时间限制
    use_cp_sat: bool = True # 使用CP-SAT求解器
    objective_scale: int = Field(default=1000, ge=1, le=1_000_000) # 目标权重

class APSJob(BaseModel):
    job_id: str # 订单ID
    product: str # 产品
    quantity: int = Field(gt=0) # 数量
    due_in_hours: int | None = Field(default=None, ge=0) # 截止时间
    profit_priority: int | None = Field(default=None, ge=0, le=100) # 利润优先级
    allowed_machines: list[str] = Field(default_factory=list) # 允许的机器

class APSMachine(BaseModel):
    machine_id: str # 机器ID
    capacity_per_hour: int = Field(gt=0) # 每小时产能
    supported_products: list[str] = Field(default_factory=list) # 支持的产品

class APSInput(BaseModel):
    jobs: list[APSJob] = Field(default_factory=list) # 订单
    machines: list[APSMachine] = Field(default_factory=list) # 机器
    planner_output: PlannerOutput # 规划输出

class PlannerOutput(BaseModel):
    intent: Literal["optimize_schedule"] = "optimize_schedule" # 规划意图
    priorities: list[OrderPriority] = Field(default_factory=list) # 优先级
    objective: ObjectiveWeights # 目标权重
    constraints: list[Constraint] = Field(default_factory=list) # 约束
    assumptions: list[str] = Field(default_factory=list) # 假设
    solver_config: SolverConfig = Field(default_factory=SolverConfig) # 求解器配置
    explanation: str = Field(min_length=10) # 解释

    @field_validator("objective", "solver_config", mode="before")
    @classmethod
    def parse_json_object_if_string(cls, v):
        if isinstance(v, str):
            v = v.strip()
            if v.startswith("{") and v.endswith("}"):
                try:
                    return json.loads(v)
                except json.JSONDecodeError:
                    pass
        return v

class ScheduledJob(BaseModel):
    job_id: str
    machine_id: str
    start_hour: int
    end_hour: int
    duration_hours: int


class SolverResult(BaseModel):
    status: str
    objective_value: int | None = None
    makespan_hours: int | None = None
    scheduled_jobs: list[ScheduledJob] = Field(default_factory=list)
    unscheduled_jobs: list[str] = Field(default_factory=list)
    notes: list[str] = Field(default_factory=list)

MCP_SERVER = MCPServerStreamableHTTP('http://localhost:8800/mcp')

PlannerAgent = Agent(
    'openrouter:xiaomi/mimo-v2-flash',
    system_prompt="""
你是一个工业APS排程Agent。你可以合理的利用工具来配合你完成任务。
将用户排程请求转换为严格的结构化输出，用于下游APS求解器。 
始终只输出可执行的约束。 
不要输出模糊的语言。 
归一化目标权重，使总和 <= 1.0 且 > 0。 
当信息缺失时使用假设，而不是编造事实。 
优先使用求解器安全的解释。
**严格输出结构化对象**：
- objective 必须是 JSON object
- solver_config 必须是 JSON object
**Example OUTPUT STRUCTURE**：
{
  "intent": "optimize_schedule",
  "objective": {
    "minimize_lateness": 0.6,
    "maximize_profit": 0.0,
    "minimize_inventory": 0.25,
    "minimize_changeover": 0.15,
    "maximize_throughput": 0.0
  },
  "solver_config": {
    "horizon_hours": 72,
    "time_limit_seconds": 10,
    "use_cp_sat": true,
    "objective_scale": 1000
  },
  "priorities": ["deadline","inventory","setup"],
  "constraints": [
    {"type":"due_date","description":"所有订单需按期完成","params":{}}
  ],
  "assumptions": ["排程时间范围为72小时"],
  "explanation": "以准时交付为主，兼顾库存和换线成本。"
}
""",
    output_type=PlannerOutput,
    toolsets=[MCP_SERVER],
    retries=4,
    output_retries=5,
)

def sort_priorities(objective: ObjectiveWeights) -> list[OrderPriority]:
    pairs = [
        (OrderPriority.deadline, objective.minimize_lateness),
        (OrderPriority.profit, objective.maximize_profit),
        (OrderPriority.inventory, objective.minimize_inventory),
        (OrderPriority.setup, objective.minimize_changeover),
        (OrderPriority.throughput, objective.maximize_throughput),
    ]
    return [key for key, value in sorted(pairs, key=lambda x: x[1], reverse=True) if value > 0]



def validate_planner_output(out: PlannerOutput) -> PlannerOutput:
    out.priorities = sort_priorities(out.objective)

    constraint_types = {c.type for c in out.constraints}

    if out.objective.minimize_lateness > 0 and ConstraintType.due_date not in constraint_types:
        out.constraints.append(
            Constraint(
                type=ConstraintType.due_date,
                description="Due-date performance should be considered because lateness is part of the objective.",
                params={"penalty_enabled": True},
            )
        )

    if out.objective.minimize_inventory > 0 and ConstraintType.inventory_limit not in constraint_types:
        out.assumptions.append(
            "Inventory optimization was requested but no explicit inventory limit was supplied; external limits may be required."
        )

    if out.solver_config.use_cp_sat and out.solver_config.objective_scale < 10:
        raise ValueError("objective_scale is too small for integer CP-SAT modeling.")

    return out

result = await PlannerAgent.run('优先准时交付，同时别让库存爆掉，换线和清洗成本也要考虑')
print(result.output.model_dump())

Logfire project URL: ]8;id=109061;https://logfire-us.pydantic.dev/evango/starter-project\https://logfire-us.pydantic.dev/evango/starter-project]8;;\

00:55:30.491 PlannerAgent run
00:55:31.294   chat xiaomi/mimo-v2-flash
00:55:32.861   running tool: get_current_time
00:55:33.128   chat xiaomi/mimo-v2-flash
{'intent': 'optimize_schedule', 'priorities': [<OrderPriority.deadline: 'deadline'>, <OrderPriority.inventory: 'inventory'>, <OrderPriority.setup: 'setup'>], 'objective': {'minimize_lateness': 0.6, 'maximize_profit': 0.0, 'minimize_inventory': 0.2, 'minimize_changeover': 0.15, 'maximize_throughput': 0.05}, 'constraints': [{'type': <ConstraintType.due_date: 'due_date'>, 'description': '所有订单必须按期完成，不允许延误', 'params': {}}, {'type': <ConstraintType.inventory_limit: 'inventory_limit'>, 'description': '各物料库存不得超过预设上限，避免爆仓', 'params': {}}, {'type': <ConstraintType.changeover: 'changeover'>, 'description': '优化换线顺序以减少切换成本和清洗需求', 'params': {}}], 'assumptions': ['排程时间范围为72小时', '库存上限基于当前物料库容设定', '换线成本包含清洗时间和材料损耗'], 'solver_config': {'horizon_hours': 72, 'time_limit_seconds': 10, 'use_cp_sat': True, 'objective_scale': 1000}, 'explanation': '以准时交付